In [ ]:

import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# -----------------------------------------------------
# SEEDS
# -----------------------------------------------------

SEEDS = [1, 7, 21, 42, 100]

N_SHOT = 5

results = []

# -----------------------------------------------------
# LOOP
# -----------------------------------------------------

for seed in SEEDS:

    print(f"\nRunning Seed = {seed}")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    # -----------------------------------------------
    # SUPPORT SET
    # -----------------------------------------------

    support_features = []
    support_labels = []

    classes = torch.unique(
        train_labels
    )

    for cls in classes:

        idx = torch.where(
            train_labels == cls
        )[0]

        idx = idx[
            torch.randperm(
                len(idx)
            )[:N_SHOT]
        ]

        support_features.append(
            train_features[idx]
        )

        support_labels.append(
            train_labels[idx]
        )

    support_features = torch.cat(
        support_features
    )

    support_labels = torch.cat(
        support_labels
    )

    # -----------------------------------------------
    # PROTOTYPES
    # -----------------------------------------------

    prototypes = []

    for cls in classes:

        proto = support_features[
            support_labels == cls
        ].mean(0)

        prototypes.append(proto)

    prototypes = torch.stack(
        prototypes
    )

    # -----------------------------------------------
    # NORMALIZATION
    # -----------------------------------------------

    proto_norm = F.normalize(
        prototypes,
        dim=1
    )

    test_norm = F.normalize(
        test_features,
        dim=1
    )

    # -----------------------------------------------
    # DISTANCE
    # -----------------------------------------------

    distances = torch.cdist(
        test_norm,
        proto_norm
    )

    preds = torch.argmin(
        distances,
        dim=1
    )

    y_true = test_labels.numpy()
    y_pred = preds.numpy()

    # -----------------------------------------------
    # METRICS
    # -----------------------------------------------

    acc = accuracy_score(
        y_true,
        y_pred
    )

    prec = precision_score(
        y_true,
        y_pred,
        average='weighted'
    )

    rec = recall_score(
        y_true,
        y_pred,
        average='weighted'
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average='weighted'
    )

    results.append([
        seed,
        acc,
        prec,
        rec,
        f1
    ])

# -----------------------------------------------------
# DATAFRAME
# -----------------------------------------------------

results_df = pd.DataFrame(
    results,
    columns=[
        "Seed",
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score"
    ]
)

# -----------------------------------------------------
# MEAN STD
# -----------------------------------------------------

mean_row = {
    "Seed": "Mean",
    "Accuracy": results_df["Accuracy"].mean(),
    "Precision": results_df["Precision"].mean(),
    "Recall": results_df["Recall"].mean(),
    "F1-Score": results_df["F1-Score"].mean()
}

std_row = {
    "Seed": "Std",
    "Accuracy": results_df["Accuracy"].std(),
    "Precision": results_df["Precision"].std(),
    "Recall": results_df["Recall"].std(),
    "F1-Score": results_df["F1-Score"].std()
}

results_df = pd.concat(
    [
        results_df,
        pd.DataFrame([mean_row]),
        pd.DataFrame([std_row])
    ],
    ignore_index=True
)

# -----------------------------------------------------
# DISPLAY
# -----------------------------------------------------

print("\n")
print("="*70)
print("MULTI-SEED EVALUATION RESULTS")
print("="*70)

display(results_df)

# -----------------------------------------------------
# SAVE CSV
# -----------------------------------------------------

results_df.to_csv(
    "multi_seed_results.csv",
    index=False
)

print(
    "\nSaved: multi_seed_results.csv"
)